In [1]:
using MAT               # pour charger .mat
using LinearAlgebra
using Distributions
using Random
using Symbolics

In [2]:
function calcul_matrice_observabilite(f_eqs, y_eqs, states)
    println("1. Calcul des Jacobiennes A et C...")
    # A = df/dx
    A = Symbolics.jacobian(f_eqs, states)
    # C = dY/dx
    C = Symbolics.jacobian(y_eqs, states)
    
    n = length(states)
    
    # Initialisation de la matrice O avec C
    O = C
    
    # Puissance courante de A (A^0, A^1, A^2...)
    A_power = I(n) # Matrice identité initiale (si besoin) ou on commence la boucle direct
    
    println("2. Construction de la matrice O (CA^k)...")
    
    # Terme courant CA^k
    current_term = C
    
    # Boucle pour empiler C, CA, CA^2 ... CA^(n-1)
    # On a déjà C, donc on itère n-1 fois
    for i in 1:(n-1)
        # On calcule le terme suivant : C * A^i
        # Pour optimiser, on peut faire : Terme_precedent * A
        # Cependant, avec Symbolics, multiplier des matrices énormes peut être lourd.
        # On recalcule A^i ou on propage. Propager est plus simple :
        
        current_term = current_term * A
        
        # Concaténation verticale (vcat)
        O = [O; current_term]
    end
    
    return O, A, C
end

calcul_matrice_observabilite (generic function with 1 method)

In [25]:
using ModelingToolkit, Symbolics

# États et paramètres
@variables  θ1 θ2 ω1 ω2 Tm1 Tm2 N
@variables PG1 PG2 θ1 θ2 PL1 PL2
@variables KL D1 D2 J1 J2 Ks ω0 a1 a2 b1 b2 Pr1 Pr2 P0

x = [θ1, θ2, ω1, ω2, Tm1, Tm2, N]  # États
u = [PG1, PG2, PL1, PL2]     # Entrées

# Sorties
F12 = KL*(θ1 - θ2)
F21 = KL*(θ2 - θ1)
PL1 = F12 - PG1
PL2 = F21 - PG2
Y = [F12, PG1, PG2]

# Système dynamique
ωr = (J1*ω1 + J2*ω2)/(J1+J2)

f = [
    ω1, 
    ω2,
    (Tm1 - (PG1/ω1) - D1*(ω1))/J1,
    (Tm2 - (PG2/ω2) - D2*(ω2))/J2,
    -a1*(Tm1 *ω0 - (N*Pr1+P0)) - b1*(ω1),
    -a2*(Tm2 *ω0 - (N*Pr2+P0)) - b2*(ω2),
    Ks*(ωr)
]

f_linear = [
    ω1, 
    ω2,
    (Tm1 - (PG1/ω0) - D1*(ω1))/J1,
    (Tm2 - (PG2/ω0) - D2*(ω2))/J2,
    -a1*(Tm1 *ω0 - (N*Pr1+P0)) - b1*(ω1),
    -a2*(Tm2 *ω0 - (N*Pr2+P0)) - b2*(ω2),
    Ks*ωr
]


7-element Vector{Num}:
                                 ω1
                                 ω2
   (Tm1 + (-PG1) / ω0 - D1*ω1) / J1
   (Tm2 + (-PG2) / ω0 - D2*ω2) / J2
 -b1*ω1 - (-P0 - N*Pr1 + Tm1*ω0)*a1
 -b2*ω2 - (-P0 - N*Pr2 + Tm2*ω0)*a2
   ((J1*ω1 + J2*ω2)*Ks) / (J1 + J2)

In [26]:


# --- EXÉCUTION ---

# Calcul
Obs_Matrix, Mat_A, Mat_C = calcul_matrice_observabilite(f_linear, Y, x)

# Affichage des dimensions pour vérifier
println("Dimensions de A : ", size(Mat_A))
println("Dimensions de C : ", size(Mat_C))
println("Dimensions de O : ", size(Obs_Matrix))

# Pour voir un élément spécifique (parce que la matrice entière est illisible en console)
print(Obs_Matrix)

valeurs_numeriques = Dict(
    # --- 1. Paramètres du Réseau & Globaux ---
    KL => 3064.0,       # Raideur pour 2 lignes (MW/rad) [cite: 113]
    Ks => 0.05,         # Gain contrôle secondaire [cite: 99]
    ω0 => 100 * π,      # Fréquence nominale 50Hz (314.159...) [cite: 97]

    # --- 2. Paramètres Générateur 1 (Table 1)  ---
    J1 => 0.4,          # Inertie (Note: valeur très faible, possiblement normalisée)
    D1 => 0.04,         # Amortissement
    Pr1 => 100.0,       # Puissance de régulation (Pr)
    a1 => 100.0,        # Gain alpha (noté "C" dans le tableau)
    b1 => 2000.0,       # Gain beta

    # --- 3. Paramètres Générateur 2 (Table 1)  ---
    J2 => 0.1,          # Inertie
    D2 => 0.02,         # Amortissement
    Pr2 => 50.0,        # Puissance de régulation (Pr)
    a2 => 100.0,        # Gain alpha
    b2 => 2000.0,       # Gain beta

    # --- 4. Point de consigne de Puissance (Attention !) ---
    # Le document a deux consignes distinctes.
    # Ton code utilisait 'P0'. Idéalement, remplace P0 par P1_0 et P2_0 dans tes équations.
    # Je les définis ici :
    P0 => 0.0,          # Valeur par défaut si ton code utilise P0 additivement
    # Si tu modifies ton modèle, utilise ces valeurs pour les consignes :
    # P1_0 => 600.0, 
    # P2_0 => 400.0,

    # --- 5. États Initiaux (Point de Fonctionnement Stable) ---
    
    # Vitesses : Nominales
    ω1 => 100 * π, 
    ω2 => 100 * π,
    
    # Angles : Calculés par l'écoulement de puissance (Load Flow)
    # P_inj1 = P_G1 (600) - P_L1 (400) = +200 MW [cite: 102, 110]
    # F12 = 200 MW. Donc 200 = KL * (θ1 - θ2) => Δθ = 200 / 3064
    θ1 => 200.0 / 3064.0, # approx 0.065 rad
    θ2 => 0.0,            # Angle de référence

    # Couples Mécaniques (Tm)
    # À l'équilibre : Tm = P_Gen / ω0 [cite: 37, 40]
    # P_Gen1 = 600, P_Gen2 = 400
    Tm1 => 600.0 / (100 * π),  # approx 1.91
    Tm2 => 400.0 / (100 * π),  # approx 1.27

    # Variable N (Contrôle secondaire)
    # À l'état stable nominal, N = 0 [cite: 58]
    N => 0.0
)

println("--- Substitution Numérique ---")

# 3. Substitution et conversion en matrice de nombres (Float64)
# On utilise 'substitute' de Symbolics, puis 'Symbolics.value' pour en faire des nombres
O_num = Symbolics.value.(substitute(Obs_Matrix, valeurs_numeriques))
A_num = Symbolics.value.(substitute(Mat_A, valeurs_numeriques))

# 4. Calcul du Rang
r = rank(O_num)
n = length(x) # Devrait être 7
println("Rang calculé : ", r)
println("Nombre d'états : ", n)

1. Calcul des Jacobiennes A et C...
2. Construction de la matrice O (CA^k)...
Dimensions de A : (7, 7)
Dimensions de C : (3, 7)
Dimensions de O : (21, 7)
Num[KL -KL 0 0 0 0 0; 0 0 0 0 0 0 0; 0 0 0 0 0 0 0; 0 0 KL -KL 0 0 0; 0 0 0 0 0 0 0; 0 0 0 0 0 0 0; 0 0 (-D1*KL) / J1 (D2*KL) / J2 KL / J1 (-KL) / J2 0; 0 0 0 0 0 0 0; 0 0 0 0 0 0 0; 0 0 ((D1^2)*KL) / (J1^2) + (-KL*b1) / J1 (-(D2^2)*KL) / (J2^2) + (KL*b2) / J2 (-KL*a1*ω0) / J1 + (-D1*KL) / (J1^2) (D2*KL) / (J2^2) + (KL*a2*ω0) / J2 (-KL*Pr2*a2) / J2 + (KL*Pr1*a1) / J1; 0 0 0 0 0 0 0; 0 0 0 0 0 0 0; 0 0 (J1*Ks*((-KL*Pr2*a2) / J2 + (KL*Pr1*a1) / J1)) / (J1 + J2) + (-D1*(((D1^2)*KL) / (J1^2) + (-KL*b1) / J1)) / J1 - b1*((-KL*a1*ω0) / J1 + (-D1*KL) / (J1^2)) (-D2*((-(D2^2)*KL) / (J2^2) + (KL*b2) / J2)) / J2 + (J2*Ks*((-KL*Pr2*a2) / J2 + (KL*Pr1*a1) / J1)) / (J1 + J2) - b2*((D2*KL) / (J2^2) + (KL*a2*ω0) / J2) (((D1^2)*KL) / (J1^2) + (-KL*b1) / J1) / J1 - a1*((-KL*a1*ω0) / J1 + (-D1*KL) / (J1^2))*ω0 ((-(D2^2)*KL) / (J2^2) + (KL*b2) / J2) / J

In [28]:
using ModelingToolkit, Symbolics

# États et paramètres
@variables dω1 dω2 dθ θ1 θ2 ω1 ω2 Tm1 Tm2 N
@variables PG1 PG2 θ1 θ2 PL1 PL2
@variables KL D1 D2 J1 J2 Ks ω0 a1 a2 b1 b2 Pr1 Pr2 P0

x = [dθ, dω1, dω2, Tm1, Tm2, N]  # États
u = [PG1, PG2]                       # Entrées

# Sorties
F12 = KL*dθ
F21 = KL*(-dθ)
PL1 = F12 - PG1
PL2 = F21 - PG2
Y = [F12, F21, PG1, PG2, PL1, PL2]

# Système dynamique
ωr = (J1*dω1 + J2*dω2)/(J1+J2)

# dω1 = ω1 - ω0
# dω2 = ω2 - ω0

f = [
    dω1 - dω2, 
    (Tm1 - (PG1/ω1) - D1*(dω1))/J1,
    (Tm2 - (PG2/ω2) - D2*(dω2))/J2,
    -a1*(Tm1 *ω0 - (N*Pr1+P0)) - b1*(dω1),
    -a2*(Tm2 *ω0 - (N*Pr2+P0)) - b2*(dω2),
    Ks*(ωr - ω0)
]

f_linear = [
    dω1 - dω2,
    (Tm1 - (PG1/ω0) - D1*(dω1))/J1,
    (Tm2 - (PG2/ω0) - D2*(dω2))/J2,
    -a1*(Tm1 *ω0 - (N*Pr1+P0)) - b1*(dω1),
    -a2*(Tm2 *ω0 - (N*Pr2+P0)) - b2*(dω2),
    Ks*ωr
]




6-element Vector{Num}:
                           dω1 - dω2
   (Tm1 + (-PG1) / ω0 - D1*dω1) / J1
   (Tm2 + (-PG2) / ω0 - D2*dω2) / J2
 -b1*dω1 - (-P0 - N*Pr1 + Tm1*ω0)*a1
 -b2*dω2 - (-P0 - N*Pr2 + Tm2*ω0)*a2
  ((J1*dω1 + J2*dω2)*Ks) / (J1 + J2)

In [29]:


# --- EXÉCUTION ---

# Calcul
Obs_Matrix, Mat_A, Mat_C = calcul_matrice_observabilite(f_linear, Y, x)

# Affichage des dimensions pour vérifier
println("Dimensions de A : ", size(Mat_A))
println("Dimensions de C : ", size(Mat_C))
println("Dimensions de O : ", size(Obs_Matrix))

# Pour voir un élément spécifique (parce que la matrice entière est illisible en console)
println("Matrix O : ", Obs_Matrix)
println("Matrix A : ", Mat_A)
println("Matrix C : ", Mat_C)

valeurs_numeriques = Dict(
    # --- 1. Paramètres du Réseau & Globaux ---
    KL => 3064.0,       # Raideur pour 2 lignes (MW/rad) [cite: 113]
    Ks => 0.05,         # Gain contrôle secondaire [cite: 99]
    ω0 => 100 * π,      # Fréquence nominale 50Hz (314.159...) [cite: 97]

    # --- 2. Paramètres Générateur 1 (Table 1)  ---
    J1 => 0.4,          # Inertie (Note: valeur très faible, possiblement normalisée)
    D1 => 0.04,         # Amortissement
    Pr1 => 100.0,       # Puissance de régulation (Pr)
    a1 => 100.0,        # Gain alpha (noté "C" dans le tableau)
    b1 => 2000.0,       # Gain beta

    # --- 3. Paramètres Générateur 2 (Table 1)  ---
    J2 => 0.1,          # Inertie
    D2 => 0.02,         # Amortissement
    Pr2 => 50.0,        # Puissance de régulation (Pr)
    a2 => 100.0,        # Gain alpha
    b2 => 2000.0,       # Gain beta

    # --- 4. Point de consigne de Puissance (Attention !) ---
    # Le document a deux consignes distinctes.
    # Ton code utilisait 'P0'. Idéalement, remplace P0 par P1_0 et P2_0 dans tes équations.
    # Je les définis ici :
    P0 => 0.0,          # Valeur par défaut si ton code utilise P0 additivement
    # Si tu modifies ton modèle, utilise ces valeurs pour les consignes :
    # P1_0 => 600.0, 
    # P2_0 => 400.0,

    # --- 5. États Initiaux (Point de Fonctionnement Stable) ---
    
    # Vitesses : Nominales
    ω1 => 100 * π, 
    ω2 => 100 * π,
    
    # Angles : Calculés par l'écoulement de puissance (Load Flow)
    # P_inj1 = P_G1 (600) - P_L1 (400) = +200 MW [cite: 102, 110]
    # F12 = 200 MW. Donc 200 = KL * (θ1 - θ2) => Δθ = 200 / 3064
    θ1 => 200.0 / 3064.0, # approx 0.065 rad
    θ2 => 0.0,            # Angle de référence

    # Couples Mécaniques (Tm)
    # À l'équilibre : Tm = P_Gen / ω0 [cite: 37, 40]
    # P_Gen1 = 600, P_Gen2 = 400
    Tm1 => 600.0 / (100 * π),  # approx 1.91
    Tm2 => 400.0 / (100 * π),  # approx 1.27

    # Variable N (Contrôle secondaire)
    # À l'état stable nominal, N = 0 [cite: 58]
    N => 0.0
)

println("--- Substitution Numérique ---")

# 3. Substitution et conversion en matrice de nombres (Float64)
# On utilise 'substitute' de Symbolics, puis 'Symbolics.value' pour en faire des nombres
O_num = Symbolics.value.(substitute(Obs_Matrix, valeurs_numeriques))
A_num = Symbolics.value.(substitute(Mat_A, valeurs_numeriques))
C_num = Symbolics.value.(substitute(Mat_C, valeurs_numeriques))

# 4. Calcul du Rang
r = rank(O_num)
n = length(x) # Devrait être 7
println("Rang calculé : ", r)
println("Nombre d'états : ", n)

1. Calcul des Jacobiennes A et C...
2. Construction de la matrice O (CA^k)...
Dimensions de A : (6, 6)
Dimensions de C : (6, 6)
Dimensions de O : (36, 6)
Matrix O : Num[KL 0 0 0 0 0; -KL 0 0 0 0 0; 0 0 0 0 0 0; 0 0 0 0 0 0; KL 0 0 0 0 0; -KL 0 0 0 0 0; 0 KL -KL 0 0 0; 0 -KL KL 0 0 0; 0 0 0 0 0 0; 0 0 0 0 0 0; 0 KL -KL 0 0 0; 0 -KL KL 0 0 0; 0 (-D1*KL) / J1 (D2*KL) / J2 KL / J1 (-KL) / J2 0; 0 (D1*KL) / J1 (-D2*KL) / J2 (-KL) / J1 KL / J2 0; 0 0 0 0 0 0; 0 0 0 0 0 0; 0 (-D1*KL) / J1 (D2*KL) / J2 KL / J1 (-KL) / J2 0; 0 (D1*KL) / J1 (-D2*KL) / J2 (-KL) / J1 KL / J2 0; 0 ((D1^2)*KL) / (J1^2) + (-KL*b1) / J1 (-(D2^2)*KL) / (J2^2) + (KL*b2) / J2 (-KL*a1*ω0) / J1 + (-D1*KL) / (J1^2) (D2*KL) / (J2^2) + (KL*a2*ω0) / J2 (-KL*Pr2*a2) / J2 + (KL*Pr1*a1) / J1; 0 (-(D1^2)*KL) / (J1^2) + (KL*b1) / J1 (-KL*b2) / J2 + ((D2^2)*KL) / (J2^2) (KL*a1*ω0) / J1 + (D1*KL) / (J1^2) (-KL*a2*ω0) / J2 + (-D2*KL) / (J2^2) (KL*Pr2*a2) / J2 + (-KL*Pr1*a1) / J1; 0 0 0 0 0 0; 0 0 0 0 0 0; 0 ((D1^2)*KL) / (J1^2) + (-KL

 Le Test de PBH (Popov-Belevitch-Hautus) 

 Mathématiquement, pour un système $\dot{x} = Ax$ avec une mesure $y = Cx$, la détectabilité se vérifie souvent via le test de PBH.
 
 Le système est détectable si et seulement si :$$\text{rang} \begin{bmatrix} A - \lambda I \\ C \end{bmatrix} = n$$

 our tout $\lambda$ (valeur propre de $A$) tel que $\text{Re}(\lambda) \ge 0$.

In [7]:
using LinearAlgebra

"""
    is_detectable(A, C; tol=1e-9)

Vérifie si la paire (A, C) est détectable. 
Version corrigée pour gérer les types abstraits.
"""
function is_detectable(A::AbstractMatrix, C::AbstractMatrix; tol=1e-9)
    n = size(A, 1)
    
    # --- CORRECTION ICI ---
    # On force la conversion en ComplexF64 pour éviter le type 'Any' ou 'Real'
    # Cela garantit que toutes les maths se font en flottant haute précision.
    A_c = ComplexF64.(A)
    C_c = ComplexF64.(C)
    # ----------------------

    eigen_values = eigvals(A_c)

    for lambda in eigen_values
        if real(lambda) >= -tol
            # Construction de la matrice PBH
            # Maintenant, A_c et lambda sont du même type précis, pas d'ambiguïté pour Julia
            pbh_matrix = [A_c - lambda * I; C_c]
            
            if rank(pbh_matrix) < n
                return false
            end
        end
    end
    
    return true
end

is_detectable

In [8]:
println("le système est détectable : ", is_detectable(A_num, C_num))

le système est détectable : true
